# Delta Pipeline Notebook

Pipeline: `dbt -> Trino -> S3(MinIO) -> read`

Only Delta format is used in this project.


## 1) Helpers

Utility helpers to run Docker/Trino commands and print outputs.


In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import trino

def detect_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Cannot locate repo root (.git not found)')

ROOT = detect_repo_root(Path.cwd().resolve())
COMPOSE_FILE = ROOT / 'infra' / 'docker-compose.trino.yml'
DBT_DIR = ROOT / 'dbt' / 'trino_pipeline'

def run_cmd(cmd: list[str], check: bool = True) -> str:
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError("Command failed (%s): %s\n%s" % (proc.returncode, ' '.join(cmd), proc.stderr))
    return proc.stdout.strip()

def trino_sql(sql: str) -> str:
    conn = trino.dbapi.connect(
        host='localhost',
        port=8080,
        user='dbt',
        catalog='delta',
        schema='analytics',
    )
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description] if cur.description else []
    lines = []
    if cols:
        lines.append(' | '.join(cols))
    lines.extend(' | '.join(str(x) for x in r) for r in rows)
    return '\n'.join(lines)


## 2) Run Pipeline

Runs `dbt seed` and `dbt run` in containers.


In [ ]:
print(run_cmd(['make', 'pipeline-run']))


## 3) Service Health

Checks container statuses.


In [ ]:
print(run_cmd(['docker', 'compose', '-f', str(COMPOSE_FILE), 'ps']))


## 4) Delta Catalog Checks

Checks catalogs, schemas, and transformed table output.


In [ ]:
print('Catalogs:')
print(trino_sql('show catalogs'))
print('\nSchemas in delta:')
print(trino_sql('show schemas from delta'))
print('\nTransformed table preview:')
print(trino_sql('select * from delta.analytics.survival_by_class order by pclass'))


## 5) Delta History

Shows Delta transaction history for observability.


In [ ]:
history_sql = '''
select version, timestamp, operation, read_version, isolation_level
from delta.analytics."survival_by_class$history"
order by version desc
limit 20
'''
print(trino_sql(history_sql))


## 6) Query Runtime Signals

Shows recent Trino queries and their states.


In [ ]:
queries_sql = '''
select query_id, state, "user", source, created
from system.runtime.queries
order by created desc
limit 15
'''
print(trino_sql(queries_sql))


## 7) dbt Artifacts

Reads `run_results.json` with per-model execution time and status.


In [ ]:
run_results = DBT_DIR / 'target' / 'run_results.json'
if run_results.exists():
    payload = json.loads(run_results.read_text(encoding='utf-8'))
    rows = []
    for r in payload.get('results', []):
        rows.append({
            'status': r.get('status'),
            'resource_type': r.get('resource_type'),
            'name': (r.get('unique_id') or '').split('.')[-1],
            'execution_time_sec': r.get('execution_time'),
        })
    display(pd.DataFrame(rows))
else:
    print('run_results.json not found. Run: make pipeline-run')


## 8) Logs (tail)

Quick tail for debugging Trino/MinIO/dbt containers.


In [ ]:
print(run_cmd([
    'docker', 'compose', '-f', str(COMPOSE_FILE),
    'logs', '--tail', '80', 'trino', 'minio', 'mc-init'
], check=False))
